# Projekt: Kundenstimmung auf Twitter verstehen

## „Kundensupport auf Twitter“ untersuchen
- mehrstufigen KI-Workflow nutzen, um die Daten eingehend zu verstehen und zu analysieren:
- mit Gemini , um den Datensatz automatisch zu erkunden und zusammenzufassen.
- mit AutoViz fort , um die Daten visuell zu verstehen.
- mit Hugging-Face-Modell verwendet , um die Stimmung in den Tweets der Kunden zu analysieren.

# Imports

In [6]:
# IMPORT

# Standard Data Science Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Automated Visualization
from autoviz.AutoViz_Class import AutoViz_Class

# Machine Learning & NLP (Offline)
import torch
from transformers import pipeline

# Local LLM Integration (Ollama / DeepSeek-Coder-V2)
import requests
import json
import ollama

# System & Resource Management
import os
import gc
import sys

# Optional: Disable Parallelism warnings for stability on Mac
#os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [7]:
# Load STRUCKTURE
# [TASK: CONFIG] Project Structure Definitions
# Definition der Projekt-Ordner
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")           # Für twcs.csv
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")       # Für Berichte/CSVs
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")       # Lokaler Cache für HF Modelle
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")           # Speziell für AutoViz PNGs
TEMP_DIR = os.path.join(PROJECT_ROOT, "temp")           # Temporäre LLM-Ablagen

# Liste der Ordner für die Erstellung
directories = [DATA_DIR, OUTPUT_DIR, MODELS_DIR, PLOTS_DIR, TEMP_DIR]

# Chirurgisches Anlegen der Ordner
for directory in directories:
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Created directory: {directory}")

# Konfiguration für Hugging Face (Offline-Cache Pfad setzen)
os.environ['HUGGINGFACE_HUB_CACHE'] = MODELS_DIR
os.environ['TRANSFORMERS_CACHE'] = MODELS_DIR

#print("\nProject structure initialized. All artifacts will be stored in sub-folders.")

In [8]:
# LOAD DATA
df = pd.read_csv("data/sample.csv")
# Kompakter Check
#print(f"Loaded: {df.shape}")
#df.head(3)

In [11]:
# [TASK: LLM_INTERFACE]
def frage(user_prompt, system_context="You are a data scientist analyzing Twitter support data."):
    """
    Nutzt lokales Ollama mit DeepSeek-Coder-V2.
    """
    response = ollama.generate(
        model='deepseek-coder-v2',
        prompt=f"{system_context}\n\nUser: {user_prompt}",
        options={
            "num_thread": 7, # Schont 1 Core für das System
            "temperature": 0 # Höchste Präzision für Analysen
        }
    )
    print(response['response'])

# --- Schritt 1: Automatische Erkundung (EDA) ---
# Wir übergeben dem LLM die Fakten direkt aus dem DataFrame
#metadata_prompt = f"""
#Analyze this dataset structure and summarize it for a presentation:
#- Columns: {df.columns.tolist()}
#- Shape: {df.shape}
#- Missing values: {df.isnull().sum().to_dict()}
#- Data Types: {df.dtypes.to_dict()}
#
#What are the 3 most important columns for sentiment analysis and why?
#"""

#print("### Gemini/DeepSeek Initial Exploration ###\n")
#frage(metadata_prompt)

## Datensatz: Kundensupport auf Twitter

### Dieser Datensatz bietet drei wesentliche Vorteile gegenüber anderen Konversationsdatensätzen:
- Fokussiert : Die Gespräche drehen sich um reale Probleme, die die Menschen gelöst haben möchten – verlorenes Gepäck, Abrechnungsprobleme, stornierte Flüge – wodurch die Daten einen klaren Zweck und eine klare Struktur erhalten.
- Natürlich : Die Sprache ist modern und wirkt authentisch, geschrieben von Menschen mit unterschiedlichem Hintergrund. Sie spiegelt wider, wie Kunden heute tatsächlich online kommunizieren.
- Kurz und bündig : Da Tweets kurz sind, wirken die Antworten authentischer und weniger einstudiert. Dies hilft Modellen, natürlicher zu lernen und unterstützt zudem eine effiziente Verarbeitung.

1. Technische Artefakte.
- Von Gemini generierte Datensatzzusammenfassungen oder Explorationsergebnisse
- AutoViz-Visualisierungen, die wichtige Muster hervorheben
- Ergebnisse der Stimmungsanalyse, die mit einem Hugging-Face-Modell erzeugt wurden
2. Analytisches Denken
- Wie Zwillinge Ihr anfängliches Verständnis und Ihre analytische Ausrichtung geprägt haben
- Was AutoViz aufdeckte, war aus den Textzusammenfassungen allein nicht ersichtlich.
- Wie die Ergebnisse der Stimmungsanalyse frühere Erkenntnisse ergänzten oder in Frage stellten
3. Überlegungen zur KI-gestützten Analyse
- Wo KI-Tools die Exploration beschleunigten oder den manuellen Aufwand reduzierten
- Wo menschliche Interpretation noch unerlässlich war
- Stärken und Schwächen der Verwendung von LLMs für die Stimmungsanalyse

Schriftliche Erläuterungen sollten als Markdown-Zellen neben den Ausgaben eingefügt werden.
Ziel ist es, Einsicht, Urteilsvermögen und den effektiven Einsatz von Werkzeugen zu demonstrieren, nicht eine erschöpfende Analyse.

1. Nutzen Sie LLM für erste Erkundungen
Laden den Datensatz, nutzen den integrierten KI-Assistenten LLM, um Ihr Projekt zu starten. Bitten Sie LLM, den Datensatz zu beschreiben und wichtige Merkmale wie Spaltennamen, Datentypen und fehlende Werte zusammenzufassen.
Diese automatisierte Analyse hilft Ihnen, die Struktur des Datensatzes schnell zu erfassen und zu entscheiden, welche Spalten für Ihre Ziele am relevantesten sind.

In [12]:
frage("Datensatz beschreiben in dem die wichtige Merkmale wie Spaltennamen, Datentypen und fehlende Werte zusammenzufassen")

 Natürlich! Um einen Datensatz zu beschreiben, der Twitter-Support-Daten enthält, ist es hilfreich, die wichtigsten Merkmale (Spaltennamen), deren Datentypen und vorhandene fehlenden Werte zusammenzufassen. Hier ist ein Beispiel für eine solche Beschreibung:

---

**Datensatzbeschreibung: Twitter Support Dataset**

Der Datensatz enthält Informationen über Unterstützungs-Tickets, die von Kunden bei Twitter erstellt wurden. Der Datensatz umfasst mehrere Spalten, die verschiedene Aspekte der Support-Interaktionen abbilden. Hier sind die wichtigsten Merkmale (Spaltennamen), deren Datentypen und eine kurze Beschreibung:

1. **Ticket ID**: Ein eindeutige Identifikationsnummer für jedes Support-Ticket.  
   - **Datentyp**: Integer (int64)
   - **Beschreibung**: Eine fortlaufende Nummer, die das Ticket eindeutig identifiziert.

2. **Erstellungsdatum**: Das Datum und die Uhrzeit, zu der das Ticket erstellt wurde.  
   - **Datentyp**: DateTime (datetime64)
   - **Beschreibung**: Gibt an, wann da

2. Visualisieren Sie die Daten mit AutoViz.
Als Nächstes auf das visuelle Verständnis der Daten. Mit AutoViz können Diagramme und Grafiken erstellen, die wichtige Erkenntnisse über die Daten liefern. Vergleichen die Ergebnisse von AutoViz mit den vorherigen Vorschlägen von LLM.

In [17]:
# [TASK: AUTOVIZ_FIXED_FINAL]
# 1. Sicherstellen, dass der Pfad absolut ist
abs_plots_dir = os.path.abspath(PLOTS_DIR)
# 2. Initialisierung des Objekts (Instanz erstellen)
AV = AutoViz_Class()
# 3. AUFRUF ÜBER DAS OBJEKT 'AV' (nicht über die Klasse)
dft = AV.AutoViz(
    filename="",
    sep=',',
    depVar='inbound',
    dfte=df,
    header=0,
    verbose=2,
    lowess=False,
    chart_format='png',
    max_rows_analyzed=5000,
    max_cols_analyzed=10,
    save_plot_dir=abs_plots_dir
)
#print(f"\nÜberprüfe jetzt diesen Ordner: {abs_plots_dir}")

Shape of your Data Set loaded: (93, 7)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
  Printing up to 30 columns (max) in each category:
    Numeric Columns : ['in_response_to_tweet_id']
    Integer-Categorical Columns: []
    String-Categorical Columns: ['author_id', 'response_tweet_id']
    Factor-Categorical Columns: []
    String-Boolean Columns: []
    Numeric-Boolean Columns: []
    Discrete String Columns: []
    NLP text Columns: ['text']
    Date Time Columns: []
    ID Columns: ['tweet_id', 'created_at']
    Columns that will not be considered in modeling: []
    6 Predictors classified...
        2 variable(s) removed since they were ID or low-information variables
        List of variables removed: ['tweet_id', 'created_at'

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
author_id,object,0.000000,45,,,No issue
text,object,0.000000,100,,,No issue
response_tweet_id,object,30.107527,69,,,"28 missing values. Impute them with mean, median, mode, or a constant value such as 123., Mixed dtypes: has 2 different data types: object, float,"
in_response_to_tweet_id,float64,26.881720,NA,119239.000000,119334.000000,"25 missing values. Impute them with mean, median, mode, or a constant value such as 123."
inbound,int64,0.000000,2,0.000000,1.000000,Target column


[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /Users/cristallagus/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     /Users/cristallagus/nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     /Users/cristallagus/nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     /Users/cristallagus/nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to
[nltk_data]    |     /Users/cristallagus/nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /Users/cristallagus/nltk_data...
[nltk_data]    

All Plots are saved in /Users/cristallagus/Desktop/GitHub/Projecten/🎭Kundenstimung auf Twitter Verstehen/output/plots/inbound
Time to run AutoViz = 8 seconds 


In [18]:
# [TASK: LLM_COMPARISON]
# Wir sammeln die Fakten für das LLM, damit es den Vergleich ziehen kann.

# 1. Fakten aus den Daten (was AutoViz visualisiert hat)
stats_summary = {
    "total_rows": len(df),
    "inbound_counts": df['inbound'].value_counts().to_dict(),
    "top_authors": df['author_id'].value_counts().head(3).to_dict(),
    "null_values": df.isnull().sum().to_dict()
}

# 2. Der Prompt für den Vergleich
vergleichs_prompt = f"""
Compare your initial EDA suggestions with the actual AutoViz results:
- Data Stats: {stats_summary}
- AutoViz classified 'text' as NLP and removed 'tweet_id'/'created_at'.

Please provide a concise comparison for my presentation:
1. What matches your initial prediction?
2. What did AutoViz reveal that wasn't obvious from the raw metadata?
3. Which 'Kundenstimmung' (Sentiment) trends should I look for in the next step?
"""

print("### Comparison: LLM vs. AutoViz Insights ###\n")
frage(vergleichs_prompt)

### Comparison: LLM vs. AutoViz Insights ###

 Certainly! Let's break down the comparison between your initial exploratory data analysis (EDA) suggestions and the results from AutoViz, as well as address the questions about sentiment trends.

### 1. Comparison with Initial EDA Suggestions
**What matches your initial prediction?**
- **Data Stats: {'total_rows': 93, 'inbound_counts': {True: 49, False: 44}, 'top_authors': {'AppleSupport': 13, 'Tesco': 8, 'SpotifyCares': 8}, 'null_values': {'tweet_id': 0, 'author_id': 0, 'inbound': 0, 'created_at': 0, 'text': 0, 'response_tweet_id': 28, 'in_response_to_tweet_id': 25}}**
  - **Initial Prediction:** You expected to see the total number of rows and counts of inbound tweets. This matches as AutoViz reports `total_rows` as 93 and provides breakdowns for inbound (`True`) and outbound (`False`) tweet counts, with 49 inbound and 44 outbound tweets.
  - **Initial Prediction:** You also expected to see the top authors based on their tweet frequency.

3. Stimmungsanalyse mithilfe von Hugging Face LLMs
Wenden abschließend ein Large Language Model (LLM) von Hugging Face an, um die textSpalte des Datensatzes zu analysieren. Diese Spalte enthält den vollständigen Tweet des Kunden. Mithilfe des Modells bewerten Sie die Stimmung der Nachrichten.
Sie können entweder Folgendes verwenden:
- Die Hugging Face Transformers-Bibliothek oder
- die Inference API ermöglicht den Zugriff, ohne Modelle herunterladen zu müssen.

In [20]:
# [TASK: SENTIMENT_ANALYSIS_GPU_HYBRID]
# 1. Dynamische Hardware-Erkennung (Chirurgische Logik)
# Erzwingt den reinen Offline-Modus für Hugging Face
os.environ['TRANSFORMERS_OFFLINE'] = "1"
os.environ['HF_HUB_OFFLINE'] = "1"
if torch.backends.mps.is_available():
    model_device = "mps" # Apple Silicon GPU
    print("🚀 Hardware-Beschleunigung: Apple Metal (MPS) erkannt.")
else:
    model_device = -1    # Standard CPU
    print("ℹ️ Hardware-Beschleunigung: Nicht verfügbar, nutze CPU.")
# 2. Initialisierung der Pipeline
print("Lade Hugging Face Sentiment-Pipeline...")
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=model_device
)
def get_sentiment_safe(text):
    """Extrahiert Stimmung mit Fehlerbehandlung."""
    try:
        # Kürzen auf 512 Zeichen für BERT-Limit
        res = classifier(str(text)[:512])[0]
        return res['label'], res['score']
    except Exception as e:
        return "NEUTRAL", 0.0
# 3. Anwendung auf den DataFrame
print(f"Analysiere {len(df)} Tweets auf {model_device}...")
results = df['text'].apply(lambda x: get_sentiment_safe(x))
# Ergebnisse in neue Spalten splitten
df['sentiment_label'] = [r[0] for r in results]
df['sentiment_score'] = [r[1] for r in results]
# 4. Speichern der Artefakte
sentiment_csv = os.path.join(OUTPUT_DIR, "sentiment_results.csv")
df.to_csv(sentiment_csv, index=False)
print(f"\n✅ Analyse abgeschlossen. Ergebnisse gespeichert in: {sentiment_csv}")
# 5. Kurze visuelle Erfolgskontrolle
display(df[['author_id', 'text', 'sentiment_label', 'sentiment_score']].head(10))
# 6. Statistische Zusammenfassung
sentiment_counts = df['sentiment_label'].value_counts()
print("\nVerteilung der Kundenstimmung:")
print(sentiment_counts)

🚀 Hardware-Beschleunigung: Apple Metal (MPS) erkannt.
Lade Hugging Face Sentiment-Pipeline...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Analysiere 93 Tweets auf mps...

✅ Analyse abgeschlossen. Ergebnisse gespeichert in: /Users/cristallagus/Desktop/GitHub/Projecten/🎭Kundenstimung auf Twitter Verstehen/output/sentiment_results.csv


,author_id,text,sentiment_label,sentiment_score
0,105834,@AppleSupport causing the reply to be disregarded and the tapped notification under the keyboard is opened😡😡😡,NEGATIVE,0.999374
1,ChaseSupport,"@105835 Your business means a lot to us. Please DM your name, zip code and additional details about your concern. ^RR https://t.co/znUu1VJn9r",POSITIVE,0.991010
2,105835,@76328 I really hope you all change but I'm sure you won't! Because you don't have to!,POSITIVE,0.936736
3,VirginTrains,"@105836 LiveChat is online at the moment - https://t.co/SY94VtU8Kq or contact 03331 031 031 option 1, 4, 3 (Leave a message) to request a call back",NEGATIVE,0.995228
4,105836,@VirginTrains see attached error message. I've tried leaving a voicemail several times in the past week https://t.co/NxVZjlYx1k,NEGATIVE,0.999260
5,VirginTrains,"@105836 Have you tried from another device, Miriam ^MM",NEGATIVE,0.985991
6,105836,"@VirginTrains yep, I've tried laptop too several times over the past week and again today. I've tried different browsers too",NEGATIVE,0.993119
7,VirginTrains,"@105836 It's working OK from here, Miriam. Does this link help https://t.co/0m2mpH15eh ? ^MM",POSITIVE,0.980761
8,105836,@VirginTrains I still haven't heard &amp; the number I'm directed to by phone is a dead end &amp; the live chat doesn't work. Can someone call me?,NEGATIVE,0.999741
9,VirginTrains,@105836 That's what we're here for Miriam 😊 The team should send you an email shortly ^HP,NEGATIVE,0.932592



Verteilung der Kundenstimmung:
NEGATIVE    67
POSITIVE    26
Name: sentiment_label, dtype: int64


# Präsentationskontext
Dieses Projekt wird im Rahmen einer Live-Präsentation vorgestellt, nachdem alle drei Projekte zur Steigerung der KI-gestützten Produktivität abgeschlossen sind.
Im Rahmen der Präsentation werden Sie Folgendes erläutern:
- Das analytische Ziel dieses Projekts
- Wie Gemini, AutoViz und Sprachmodelle verwendet wurden
- Wichtige Erkenntnisse zur Stimmungslage aufgedeckt
- Was Sie über die Verwendung von KI für textbasierte Analysen gelernt haben

Der Schwerpunkt der Präsentation liegt auf der Interpretation und der Steigerung der Produktivität, nicht auf technischen Details der Umsetzung.

In [22]:
# [TASK: FINAL_QUALITATIVE_ANALYSIS]
import pandas as pd

# Daten für die Interpretation vorbereiten
total = len(df)
pos = len(df[df['sentiment_label'] == 'POSITIVE'])
neg = len(df[df['sentiment_label'] == 'NEGATIVE'])
top_company = df[df['inbound'] == False]['author_id'].value_counts().idxmax()

# Der präzise Prompt für die Qualitätssicherung
quality_prompt = f"""
Analyze the following results for a formal project reflection:

1. DATA INSIGHTS:
- Total tweets analyzed: {total}
- Sentiment: {pos} Positive vs. {neg} Negative.
- Most active Support-Channel: {top_company}

2. TOOLCHAIN REFLECTION:
- Step 1: LLM-based Metadata EDA (DeepSeek-Coder-V2)
- Step 2: Automated Visual Exploration (AutoViz)
- Step 3: GPU-accelerated Sentiment Classification (Hugging Face DistilBERT on MPS)

TASK:
Write a reflection that meets these criteria:
- INTERPRETATION: What does the ratio of {pos}/{neg} tell us about customer expectations in Twitter support?
- TOOL EVALUATION: How did the combination of AutoViz (visual) and DistilBERT (textual) provide a deeper understanding than just looking at the CSV?
- PRODUCTIVITY: Explain the advantage of using local GPU (MPS) and local LLMs for data privacy and speed.
- CONCLUSION: Give one actionable advice for {top_company} based on these sentiment numbers.
"""

print("### Generiere abschließende Projekt-Reflexion ###\n")
frage(quality_prompt)

### Generiere abschließende Projekt-Reflexion ###

 ### Project Reflection: Twitter Support Data Analysis

#### INTERPRETATION: What does the ratio of 26/67 tell us about customer expectations in Twitter support?
The data reveals a significant imbalance between positive and negative sentiments, with 26 positive tweets against 67 negative ones. This stark contrast suggests that customers are highly critical or dissatisfied with their experiences on Twitter through AppleSupport. The prevalence of negativity indicates unmet expectations or poor service quality, which is crucial for Apple to address promptly. It highlights the importance of proactive engagement and effective resolution mechanisms in managing customer expectations on social media platforms like Twitter.

#### TOOL EVALUATION: How did the combination of AutoViz (visual) and DistilBERT (textual) provide a deeper understanding than just looking at the CSV?
The integration of visual exploration tools (AutoViz) with textual anal

# Qualitätserwartungen
- KI-Werkzeuge sollten gezielt und nicht oberflächlich eingesetzt werden.
- Die Ergebnisse müssen klar interpretiert und in den Kontext gesetzt werden.
- Die Reflexionen sollten ein Verständnis sowohl der Daten als auch der Werkzeuge erkennen lassen.
- Die gewonnenen Erkenntnisse sollten auf den im Rahmen der Analyse gewonnenen Erkenntnissen beruhen.

# Antwort

### Die Analyse zeigt ein deutliches Übergewicht an negativer Stimmung (72 % negative Tweets). Dies verdeutlicht, dass Twitter in diesem Datensatz primär als Beschwerdekanal und nicht als allgemeiner Kommunikationskanal genutzt wird.